In [ ]:
TARGET_COLUMN = 'UBER'  # Stock ticker for Viatris

# Step 1: Import Libraries and Install Dependencies
print("\nStep 1: Installing dependencies...")
!pip install --force-reinstall tensorflow==2.16.1 numpy==2.0.2 pandas==2.2.2 scikit-learn==1.5.2 optuna==4.0.0 lightgbm==4.5.0 xgboost==2.1.1 pywt==1.5.0 pykalman==0.9.7 transformers==4.44.2 torch==2.4.1 requests==2.32.3 beautifulsoup4==4.12.3
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For CUDA debugging
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from google.colab import files
import warnings
warnings.filterwarnings('ignore')
import time
import random
import torch


Step 1: Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.9 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement pywt==1.5.0 (from versions: none)
ERROR: No matching distribution found for pywt==1.5.0


In [ ]:
# Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True
    print("Optuna version:", optuna.__version__)
except Exception:
    print("Optuna installation failed.")
    !pip install optuna==4.0.0
    import optuna
    from optuna.samplers import TPESampler
    OPTUNA_AVAILABLE = True

# TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, GRU, Dropout, MultiHeadAttention, GlobalAveragePooling1D, Input, Flatten
    from tensorflow.keras.regularizers import l2
    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception:
    print("TensorFlow installation failed.")
    !pip install tensorflow==2.16.1
    import tensorflow as tf
    TF_AVAILABLE = True

# PyWavelets
try:
    import pywt
    PYWT_AVAILABLE = True
except Exception:
    print("PyWavelets installation failed.")
    !pip install pywt==1.5.0
    import pywt
    PYWT_AVAILABLE = True

# PyKalman
try:
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True
except Exception:
    print("PyKalman installation failed.")
    !pip install pykalman==0.9.7
    from pykalman import KalmanFilter
    PYKALMAN_AVAILABLE = True

# LightGBM
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

# Transformers and PyTorch
TRANSFORMERS_AVAILABLE = False
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True
    print("Transformers version:", transformers.__version__)
    print("PyTorch version:", torch.__version__)
except Exception:
    print("Transformers or PyTorch installation failed.")
    !pip install transformers==4.44.2 torch==2.4.1
    import transformers
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    TRANSFORMERS_AVAILABLE = True

# Requests and BeautifulSoup
import requests
from bs4 import BeautifulSoup

Optuna installation failed.
  Using cached optuna-4.0.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.8/362.8 kB 9.9 MB/s eta 0:00:00
TensorFlow version: 2.19.0
PyKalman installation failed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 8.8 MB/s eta 0:00:00
Transformers version: 4.57.3
PyTorch version: 2.9.0+cu126


In [ ]:
# Step 2: Load and Clean Data
print("\nStep 2: Loading and cleaning data...")
try:
    uploaded = files.upload()
    stock_data = {}
    news_data = None
    for file_name in uploaded.keys():
        print(f"Processing file: {file_name}")
        if file_name.endswith('.csv'):
            if 'news_data' in file_name.lower():
                try:
                    news_data = pd.read_csv(file_name)
                    print(f"News data loaded: {len(news_data)} rows")
                    print("News data columns:", news_data.columns.tolist())
                    print("News data sample:\n", news_data.head(2))
                except Exception as e:
                    print(f"Error loading news data {file_name}: {e}")
                    news_data = None
            else:
                try:
                    df = pd.read_csv(file_name)
                    print(f"Columns: {df.columns.tolist()}, Rows: {len(df)}")
                    if 'Timestamp' not in df.columns or TARGET_COLUMN not in df.columns:
                        raise ValueError(f"Expected 'Timestamp' and '{TARGET_COLUMN}' columns")
                    stock_data[TARGET_COLUMN] = df
                    stock_data[TARGET_COLUMN][TARGET_COLUMN] = stock_data[TARGET_COLUMN][TARGET_COLUMN].interpolate().ffill().bfill()
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna()
                    stock_data[TARGET_COLUMN]['Timestamp'] = pd.to_datetime(stock_data[TARGET_COLUMN]['Timestamp'], errors='coerce')
                    stock_data[TARGET_COLUMN] = stock_data[TARGET_COLUMN].dropna(subset=['Timestamp'])
                    print(f"Rows after cleaning: {len(stock_data[TARGET_COLUMN])}")
                    print("Price data sample:\n", stock_data[TARGET_COLUMN].head(2))
                except Exception as e:
                    print(f"Error loading price data {file_name}: {e}")
except Exception as e:
    print(f"Error during file upload: {e}")
    exit()


Step 2: Loading and cleaning data...


Saving UBER.csv to UBER.csv
Processing file: UBER.csv
Columns: ['Timestamp', 'UBER'], Rows: 7412
Rows after cleaning: 7412
Price data sample:
                   Timestamp       UBER
0 2025-11-26 14:30:00+00:00  84.114998
1 2025-11-26 14:31:00+00:00  83.903801


In [ ]:
# Step 2.5: Fetch News from FinViz if No News Data Provided
if news_data is None and TRANSFORMERS_AVAILABLE:
    print("\nStep 2.5: No news data uploaded. Fetching news from FinViz...")
    def fetch_finviz_news(ticker):
        url = f'https://finviz.com/quote.ashx?t={ticker}'
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            news_table = soup.find('table', {'id': 'news-table'})
            if not news_table:
                print("No news table found on FinViz")
                return None
            news_data = []
            for row in news_table.findAll('tr'):
                cells = row.findAll('td')
                if len(cells) < 2:
                    continue
                headline_link = cells[1].find('a')
                if not headline_link:
                    continue
                headline = headline_link.text.strip()
                link = headline_link['href']
                if link.startswith('/'):
                    link = f'https://finviz.com{link}'
                news_data.append({'Headline': headline, 'Headline_Link': link})
                time.sleep(0.5)
            df = pd.DataFrame(news_data)
            df = df.head(100)  # Limit to last 100 articles
            df['News_Index'] = range(1, len(df) + 1)  # 1 (least recent) to 100 (most recent)
            print(f"Fetched {len(df)} news articles from FinViz")
            return df
        except Exception as e:
            print(f"Error fetching FinViz data for {ticker}: {e}")
            return None

    def fetch_article_content(url, headline):
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.114 Safari/537.36'
        ]
        for attempt in range(3):
            try:
                headers = {'User-Agent': random.choice(user_agents)}
                response = requests.get(url, headers=headers, timeout=10)
                response.raise_for_status()
                soup = BeautifulSoup(response.text, 'html.parser')
                paragraphs = soup.find_all('p')
                article_text = ' '.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                if len(article_text.strip()) > 50:
                    print(f"Successfully fetched article from {url[:50]}...")
                    return article_text
                else:
                    print(f"Short content from {url[:50]}...")
                    return headline
            except Exception as e:
                print(f"Error fetching article from {url[:50]}...: {e}")
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    return headline
        return headline

    news_data = fetch_finviz_news(TARGET_COLUMN)
    if news_data is not None and not news_data.empty:
        print(f"Fetching article content for {len(news_data)} articles...")
        news_data['News_detail'] = news_data.apply(lambda row: fetch_article_content(row['Headline_Link'], row['Headline']), axis=1)
        news_data['News_detail'] = news_data['News_detail'].apply(lambda x: str(x) if x and len(str(x).strip()) > 0 else None)
        valid_articles = news_data['News_detail'].notna().sum()
        print(f"Fetched {valid_articles} valid articles out of {len(news_data)}")
        if valid_articles == 0:
            print("No valid article content fetched. Disabling sentiment analysis.")
            news_data = None
    else:
        print("Failed to fetch news from FinViz.")


Step 2.5: No news data uploaded. Fetching news from FinViz...
Fetched 100 news articles from FinViz
Fetching article content for 100 articles...
Successfully fetched article from https://finviz.com/news/262333/the-zacks-analyst-b...
Successfully fetched article from https://finviz.com/news/262082/3-reasons-to-buy-ub...
Successfully fetched article from https://finviz.com/news/261921/can-jobys-25-vertip...
Successfully fetched article from https://finviz.com/news/261403/should-you-invest-1...
Successfully fetched article from https://finance.yahoo.com/news/billionaire-mark-cu...
Successfully fetched article from https://finance.yahoo.com/news/founder-just-landed...
Successfully fetched article from https://finviz.com/news/261111/best-stock-to-buy-r...
Successfully fetched article from https://finviz.com/news/261179/will-serve-robotics...
Successfully fetched article from https://finance.yahoo.com/m/e2d02c37-d865-3460-bda...
Successfully fetched article from https://finviz.com/news/2611

In [ ]:
# Step 2.6: Process News Data with FinBERT
if TRANSFORMERS_AVAILABLE and news_data is not None:
    print("\nStep 2.6: Processing news data with FinBERT...")
    tokenizer = None
    model = None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
    try:
        torch.cuda.empty_cache()
        tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        model.to(device)
        print(f"FinBERT model loaded on {device}.")
    except Exception as e:
        print(f"Failed to load FinBERT model: {e}. Disabling sentiment analysis.")
        TRANSFORMERS_AVAILABLE = False
        news_data = None

    if news_data is not None:
        print("Summarizing news articles...")
        summarizer = None
        try:
            torch.cuda.empty_cache()
            # Using device=-1 for CPU, which is the default for summarization pipeline
            summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)
            print("Summarizer loaded on CPU.")
        except Exception as e:
            print(f"Failed to load summarizer: {e}")
            # Fallback to using headline if summarizer fails
            news_data['Summary'] = news_data['News_detail'].fillna(news_data['Headline'])

        if summarizer:
            def summarize_text(text):
                if not text or pd.isna(text) or len(str(text).strip()) == 0:
                    return ""
                text = str(text)[:1024] # Truncate to avoid exceeding max token limit
                word_count = len(text.split())
                if word_count <= 30: # Summarize only if text is long enough
                    return text
                try:
                    # Adjust max_length and min_length based on input text length
                    max_len = min(150, max(30, int(word_count * 0.3)))
                    min_len = max(10, min(max_len, int(word_count * 0.1)))
                    summary = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False, batch_size=1)[0]['summary_text']
                    return summary
                except Exception as e:
                    print(f"Error summarizing text {text[:50]}...: {e}")
                    return text[:256] # Return truncated text on error

            # Apply summarization only to rows where News_detail is not None and not empty
            news_data['Summary'] = news_data.apply(
                lambda row: summarize_text(row['News_detail']) if pd.notna(row['News_detail']) and len(str(row['News_detail']).strip()) > 0 else row['Headline'],
                axis=1
            )

            unique_summaries = news_data['Summary'].nunique()
            print(f"Generated {unique_summaries} unique summaries out of {len(news_data)}")
            print("News summaries sample:\n", news_data[['Summary']].head(5))

        def get_finbert_sentiment(text):
            if not text or pd.isna(text) or len(str(text).strip()) == 0:
                return np.array([0.33, 0.33, 0.33]) # Return neutral scores for empty or invalid text
            try:
                # Use the text directly for sentiment analysis, truncation handled by tokenizer
                inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
                with torch.no_grad():
                    outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
                return probs.cpu().numpy().flatten()
            except Exception as e:
                print(f"Error analyzing sentiment for text {str(text)[:50]}...: {e}")
                return np.array([0.33, 0.33, 0.33]) # Return neutral scores on error

        # Apply sentiment analysis to the 'Summary' column
        news_data['sentiment_scores'] = news_data['Summary'].apply(get_finbert_sentiment)

        # Expand the sentiment scores into separate columns
        news_data[['positive', 'negative', 'neutral']] = pd.DataFrame(
            news_data['sentiment_scores'].tolist(),
            columns=['positive', 'negative', 'neutral'],
            index=news_data.index
        )
        news_data['polarity'] = news_data['positive'] - news_data['negative']
        news_data['strength'] = news_data[['positive', 'negative', 'neutral']].max(axis=1)
        news_data['weight'] = news_data['News_Index'] / news_data['News_Index'].max() if news_data['News_Index'].max() > 0 else 0 # Normalize weight
        news_data['impact'] = news_data['polarity'] * news_data['strength'] * news_data['weight']
        print("Sentiment scores and impact sample:\n", news_data[['News_Index', 'positive', 'negative', 'neutral', 'polarity', 'strength', 'weight', 'impact']].head(5))

        # Merge news impact with price data
        try:
            if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
                print("Aligning news with price data...")
                stock_df = stock_data[TARGET_COLUMN].sort_values('Timestamp').reset_index(drop=True)
                n_news = len(news_data)
                n_prices = len(stock_df)

                # Ensure news_data has the same number of rows as stock_df for direct merging
                if n_news != n_prices:
                    print(f"Mismatch between news ({n_news}) and price ({n_prices}) data rows. Resampling news data...")
                    # Create a dummy DataFrame for news with the same index as stock_df
                    aligned_news_data = pd.DataFrame(index=stock_df.index, columns=news_data.columns)

                    # Assign news data to the aligned DataFrame, repeating news if necessary
                    news_indices = np.arange(n_news)
                    aligned_news_data.iloc[:min(n_news, n_prices)] = news_data.iloc[:min(n_news, n_prices)].values

                    # If there are more price data points than news, forward fill with the last news sentiment
                    if n_prices > n_news:
                         aligned_news_data.iloc[n_news:] = news_data.iloc[-1].values

                    # If there are more news data points than price data points, truncate news
                    if n_news > n_prices:
                         aligned_news_data = aligned_news_data.iloc[:n_prices]

                    news_data = aligned_news_data

                # Ensure columns match before assignment
                news_sentiment_cols = ['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact']
                if all(col in news_data.columns for col in news_sentiment_cols):
                    merged_data = stock_df.copy()
                    merged_data[news_sentiment_cols] = news_data[news_sentiment_cols].values
                    # Fill any remaining NaNs after alignment with neutral sentiment
                    merged_data[news_sentiment_cols] = merged_data[news_sentiment_cols].fillna(0.33)
                    stock_data[TARGET_COLUMN] = merged_data
                    print(f"Merged data shape: {stock_data[TARGET_COLUMN].shape}")
                    print("Merged data sample:\n", stock_data[TARGET_COLUMN].head(5))
                else:
                     print("Sentiment columns not found in news data after alignment. Proceeding without sentiment features.")

        except Exception as e:
            print(f"Error merging news and price data: {e}. Proceeding without sentiment features.")
            news_data = None
else:
    print("No news data or Transformers unavailable.")


Step 2.6: Processing news data with FinBERT...


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Summarizer loaded on CPU.
Generated 95 unique summaries out of 100
News summaries sample:
                                              Summary
0  Zacks Equity Research analysts discuss the lat...
1  Uber Technologies (NYSE: UBER) has long grabbe...
2  JOBY Aviation has teamed up with Metropolis Te...
3  Uber Technologies (NYSE: UBER) is the defining...
4  The self-made billionaire drew a line between ...
Sentiment scores and impact sample:
    News_Index  positive  negative   neutral  polarity  strength  weight  \
0           1  0.020570  0.054556  0.924874 -0.033986  0.924874    0.01   
1           2  0.258816  0.011009  0.730175  0.247807  0.730175    0.02   
2           3  0.796298  0.006646  0.197056  0.789652  0.796298    0.03   
3           4  0.509188  0.081832  0.408981  0.427356  0.509188    0.04   
4           5  0.150091  0.038634  0.811275  0.111456  0.811275    0.05   

     impact  
0 -0.000314  
1  0.003619  
2  0.018864  
3  0.008704  
4  0.004521  
Aligning news with 

In [ ]:
# Step 3: Denoise Data for Highest SNR
print("\nStep 3: Denoising data...")
price_scaler = MinMaxScaler()
feature_columns = [TARGET_COLUMN]
if TRANSFORMERS_AVAILABLE and news_data is not None:
    feature_columns.extend(['positive', 'negative', 'neutral', 'polarity', 'strength', 'impact'])
if TARGET_COLUMN in stock_data and not stock_data[TARGET_COLUMN].empty:
    scaled_data = price_scaler.fit_transform(stock_data[TARGET_COLUMN][[TARGET_COLUMN]])
    print(f"Scaled data shape: {scaled_data.shape}")

    def calculate_snr(original, denoised):
        noise = original - denoised
        signal_power = np.mean(original ** 2)
        noise_power = np.var(noise)
        return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

    def optimize_denoising(trial, data):
        methods = []
        if PYWT_AVAILABLE:
            wavelet = trial.suggest_categorical('wavelet', ['db1', 'db2', 'sym2'])
            level = trial.suggest_int('wavelet_level', 1, 3)
            wavelet_denoised = pywt.wavedec(data, wavelet, level=level)
            threshold = np.std(wavelet_denoised[-1]) * np.sqrt(2 * np.log(len(data)))
            wavelet_denoised = pywt.waverec([pywt.threshold(c, threshold, mode='soft') for c in wavelet_denoised], wavelet)[:len(data)]
            snr_wavelet = calculate_snr(data, wavelet_denoised)
            methods.append(('wavelet', snr_wavelet, wavelet_denoised))
        ma_window = trial.suggest_int('ma_window', 3, 10)
        ma_denoised = pd.Series(data).rolling(window=ma_window, min_periods=1).mean().values
        snr_ma = calculate_snr(data, ma_denoised)
        methods.append(('ma', snr_ma, ma_denoised))
        if TF_AVAILABLE:
            encoding_dim = trial.suggest_int('encoding_dim', 16, 32)
            model = Sequential([Dense(encoding_dim, activation='relu', input_shape=(1,)),
                                Dense(1, activation='linear')])
            model.compile(optimizer='adam', loss='mse')
            autoencoder_denoised = model.predict(data.reshape(-1, 1), verbose=0).flatten()
            snr_autoencoder = calculate_snr(data, autoencoder_denoised)
            methods.append(('autoencoder', snr_autoencoder, autoencoder_denoised))
        if PYKALMAN_AVAILABLE:
            kf = KalmanFilter(transition_matrices=[1], observation_matrices=[1], initial_state_mean=data[0])
            kalman_denoised, _ = kf.filter(data)
            snr_kalman = calculate_snr(data, kalman_denoised.flatten())
            methods.append(('kalman', snr_kalman, kalman_denoised))
        best_method = max(methods, key=lambda x: x[1] if not np.isinf(x[1]) else -float('inf'))
        return best_method[1], best_method[2]

    if OPTUNA_AVAILABLE:
        print("Optimizing denoising with Optuna...")
        study_denoise = optuna.create_study(direction='maximize', sampler=TPESampler())
        study_denoise.optimize(lambda trial: optimize_denoising(trial, scaled_data[:, 0])[0], n_trials=5)
        _, denoised_data = optimize_denoising(optuna.trial.FixedTrial(study_denoise.best_params), scaled_data[:, 0])
        print(f"Best SNR after Optuna: {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")
    else:
        print("Optuna unavailable. Using default moving average denoising...")
        denoised_data = pd.Series(scaled_data[:, 0]).rolling(window=5, min_periods=1).mean().values
        print(f"Best SNR (default): {calculate_snr(scaled_data[:, 0], denoised_data):.2f} dB")

    if TRANSFORMERS_AVAILABLE and news_data is not None:
        stock_denoised_data = {TARGET_COLUMN: stock_data[TARGET_COLUMN][feature_columns].copy()}
        stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN] = denoised_data
    else:
        stock_denoised_data = {TARGET_COLUMN: pd.DataFrame({TARGET_COLUMN: denoised_data})}
else:
    print("No stock data available. Exiting.")
    exit()

[I 2025-12-26 12:15:36,229] A new study created in memory with name: no-name-e4416154-5c2e-4eda-b20f-b88c8fe88066



Step 3: Denoising data...
Scaled data shape: (7412, 1)
Optimizing denoising with Optuna...


[I 2025-12-26 12:15:40,993] Trial 0 finished with value: 48.52356667841681 and parameters: {'wavelet': 'sym2', 'wavelet_level': 1, 'ma_window': 9, 'encoding_dim': 25}. Best is trial 0 with value: 48.52356667841681.
[I 2025-12-26 12:15:43,069] Trial 1 finished with value: 47.12581188323884 and parameters: {'wavelet': 'db2', 'wavelet_level': 3, 'ma_window': 6, 'encoding_dim': 18}. Best is trial 0 with value: 48.52356667841681.
[I 2025-12-26 12:15:45,131] Trial 2 finished with value: 47.12581188323884 and parameters: {'wavelet': 'db2', 'wavelet_level': 2, 'ma_window': 3, 'encoding_dim': 16}. Best is trial 0 with value: 48.52356667841681.
[I 2025-12-26 12:15:47,199] Trial 3 finished with value: 47.12581188323884 and parameters: {'wavelet': 'db2', 'wavelet_level': 2, 'ma_window': 8, 'encoding_dim': 16}. Best is trial 0 with value: 48.52356667841681.
[I 2025-12-26 12:15:50,101] Trial 4 finished with value: 47.12581188323884 and parameters: {'wavelet': 'sym2', 'wavelet_level': 3, 'ma_window':

Best SNR after Optuna: 48.52 dB


In [ ]:
# Step 4: Prepare Sequences
print("\nStep 4: Preparing sequences...")
SEQ_LENGTH = 30
WINDOW_SIZE = 300
stock_X_deep = {}
stock_X_ml = {}
stock_y = {}
full_data = stock_denoised_data[TARGET_COLUMN][feature_columns].values
X_for_prediction = np.array([full_data[i:i + SEQ_LENGTH] for i in range(len(full_data) - SEQ_LENGTH)])
stock_y[TARGET_COLUMN] = stock_denoised_data[TARGET_COLUMN][TARGET_COLUMN].values[SEQ_LENGTH:]
stock_X_deep[TARGET_COLUMN] = X_for_prediction
stock_X_ml[TARGET_COLUMN] = X_for_prediction
print(f"X_deep shape: {stock_X_deep[TARGET_COLUMN].shape}, y shape: {stock_y[TARGET_COLUMN].shape}")
last_300_scaled = stock_denoised_data[TARGET_COLUMN][feature_columns].values[-WINDOW_SIZE:]

# Define fixed splits for training, validation, and testing
m = len(stock_y[TARGET_COLUMN])
val_size = 300
test_size = 300
if m < val_size + test_size + SEQ_LENGTH:
    print("Data too small for requested splits. Reducing val and test sizes.")
    val_size = max(1, m // 4)
    test_size = max(1, m // 4)
train_size = m - val_size - test_size
if train_size < 1:
    print("Insufficient data for training. Exiting.")
    exit()

train_X = stock_X_deep[TARGET_COLUMN][:train_size]
train_y = stock_y[TARGET_COLUMN][:train_size]
val_X = stock_X_deep[TARGET_COLUMN][train_size:train_size + val_size]
val_y = stock_y[TARGET_COLUMN][train_size:train_size + val_size]
test_X = stock_X_deep[TARGET_COLUMN][train_size + val_size:]
test_y = stock_y[TARGET_COLUMN][train_size + val_size:]
print(f"Train size: {train_size}, Val size: {val_size}, Test size: {test_size}")


Step 4: Preparing sequences...
X_deep shape: (7382, 30, 7), y shape: (7382,)
Train size: 6782, Val size: 300, Test size: 300


In [ ]:
# Step 5: Define and Optimize Models
print("\nStep 5: Defining and optimizing models...")
feature_dim = len(feature_columns)
def optimize_cnn_lstm(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'filters': trial.suggest_int('filters', 32, 64),
            'lstm_units': trial.suggest_int('lstm_units', 50, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    filters = params['filters']
    lstm_units = params['lstm_units']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_conv1d_lstm(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'filters': trial.suggest_int('filters', 32, 64),
            'lstm_units': trial.suggest_int('lstm_units', 50, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    filters = params['filters']
    lstm_units = params['lstm_units']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Conv1D(filters=filters, kernel_size=3, padding='same', activation='relu', kernel_regularizer=l2(0.02)),
        LSTM(lstm_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_bilstm(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'lstm_units': trial.suggest_int('lstm_units', 50, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    lstm_units = params['lstm_units']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        Bidirectional(LSTM(lstm_units, kernel_regularizer=l2(0.02))),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_stacked_lstm(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'lstm_units': trial.suggest_int('lstm_units', 50, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    lstm_units = params['lstm_units']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        LSTM(lstm_units // 2, kernel_regularizer=l2(0.02)),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_gru(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'gru_units': trial.suggest_int('gru_units', 50, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    gru_units = params['gru_units']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    model = Sequential([
        Input(shape=(SEQ_LENGTH, feature_dim)),
        GRU(gru_units, kernel_regularizer=l2(0.02)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_lightgbm(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 1.0)
        }
    return lgb.LGBMRegressor(**params, force_col_wise=True)

def optimize_xgboost(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-1),
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0)
        }
    return xgb.XGBRegressor(**params)

def optimize_transformer(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'num_heads': trial.suggest_int('num_heads', 4, 8),
            'ff_dim': trial.suggest_int('ff_dim', 64, 128),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    num_heads = params['num_heads']
    ff_dim = params['ff_dim']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(inputs, inputs)
    x = Dropout(dropout_rate)(x)
    x = MultiHeadAttention(num_heads=num_heads, key_dim=SEQ_LENGTH // num_heads)(x, x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(ff_dim, activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

def optimize_tcn(trial_or_params):
    if isinstance(trial_or_params, dict):
        params = trial_or_params
    else:
        trial = trial_or_params
        params = {
            'filters': trial.suggest_int('filters', 32, 64),
            'kernel_size': trial.suggest_int('kernel_size', 2, 5),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.3, 0.6),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3)
        }
    filters = params['filters']
    kernel_size = params['kernel_size']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    inputs = Input(shape=(SEQ_LENGTH, feature_dim))
    x = inputs
    shortcut = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    x = Dropout(dropout_rate)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu', kernel_regularizer=l2(0.02))(x)
    shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(shortcut)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.Activation('relu')(x)
    x = Flatten()(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(0.02))(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

models = [
    ('cnn_lstm', optimize_cnn_lstm, 'deep'),
    ('conv1d_lstm', optimize_conv1d_lstm, 'deep'),
    ('bilstm', optimize_bilstm, 'deep'),
    ('stacked_lstm', optimize_stacked_lstm, 'deep'),
    ('gru', optimize_gru, 'deep'),
    ('lightgbm', optimize_lightgbm, 'ml'),
    ('xgboost', optimize_xgboost, 'ml'),
    ('transformer', optimize_transformer, 'deep'),
    ('tcn', optimize_tcn, 'deep')
]


Step 5: Defining and optimizing models...


In [ ]:
# Step 6: Train and Evaluate Models with Fixed Splits
print("\nStep 6: Training and evaluating models...")
def calculate_adj_r2(y_true_inv, y_pred_inv, n_predictors):
    ss_res = np.sum((y_true_inv - y_pred_inv) ** 2)
    ss_tot = np.sum((y_true_inv - np.mean(y_true_inv)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    n = len(y_true_inv)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1) if n > n_predictors + 1 else 0
    return adj_r2

stock_results = {}
for stock_name in stock_data.keys():
    results = {}
    for model_name, model_fn, data_type in models:
        if not (TF_AVAILABLE and data_type == 'deep') and not (LGB_AVAILABLE and model_name == 'lightgbm') and not (XGB_AVAILABLE and model_name == 'xgboost'):
            continue
        print(f"\nTraining {model_name} for {stock_name}...")
        try:
            best_params = {}
            val_adj_r2 = 0
            if OPTUNA_AVAILABLE:
                study = optuna.create_study(direction='minimize', sampler=TPESampler())
                def objective(trial):
                    model = model_fn(trial)
                    if data_type == 'ml':
                        train_X_2d = train_X.reshape(train_X.shape[0], -1)
                        val_X_2d = val_X.reshape(val_X.shape[0], -1)
                        model.fit(train_X_2d, train_y)
                        preds = model.predict(val_X_2d)
                    else:
                        model.fit(train_X, train_y, epochs=100, batch_size=64, validation_data=(val_X, val_y), verbose=0,
                                  callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])
                        preds = model.predict(val_X, verbose=0).flatten()
                    val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                    preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                    adj_r2 = calculate_adj_r2(val_y_inv, preds_inv, feature_dim)
                    return -adj_r2
                study.optimize(objective, n_trials=10)
                val_adj_r2 = -study.best_value
                best_params = study.best_params
            else:
                model = model_fn({})
                if data_type == 'ml':
                    train_X_2d = train_X.reshape(train_X.shape[0], -1)
                    val_X_2d = val_X.reshape(val_X.shape[0], -1)
                    model.fit(train_X_2d, train_y)
                    preds = model.predict(val_X_2d)
                else:
                    model.fit(train_X, train_y, epochs=100, batch_size=64, validation_data=(val_X, val_y), verbose=0,
                              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])
                    preds = model.predict(val_X, verbose=0).flatten()
                val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
                preds_inv = price_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
                val_adj_r2 = calculate_adj_r2(val_y_inv, preds_inv, feature_dim)

            # Build final model with best params
            model = model_fn(best_params) if OPTUNA_AVAILABLE else model_fn({})

            # Fit final model on train + val
            train_val_X = np.concatenate((train_X, val_X), axis=0)
            train_val_y = np.concatenate((train_y, val_y))
            if data_type == 'ml':
                train_val_X_2d = train_val_X.reshape(train_val_X.shape[0], -1)
                model.fit(train_val_X_2d, train_val_y)
            else:
                model.fit(train_val_X, train_val_y, epochs=100, batch_size=64, verbose=0,
                          callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)])

            # Evaluate on test
            if data_type == 'ml':
                test_X_2d = test_X.reshape(test_X.shape[0], -1)
                test_preds = model.predict(test_X_2d)
            else:
                test_preds = model.predict(test_X, verbose=0).flatten()
            test_y_inv = price_scaler.inverse_transform(test_y.reshape(-1, 1)).flatten()
            test_preds_inv = price_scaler.inverse_transform(test_preds.reshape(-1, 1)).flatten()
            test_adj_r2 = calculate_adj_r2(test_y_inv, test_preds_inv, feature_dim)

            results[model_name] = {'adj_r2': test_adj_r2, 'best_model': model, 'val_adj_r2': val_adj_r2, 'best_params': best_params}
            print(f"{model_name} - {stock_name} - Val Adj R²: {val_adj_r2:.6f}, Test Adj R²: {test_adj_r2:.6f}")
        except Exception as e:
            print(f"Error training {model_name} for {stock_name}: {e}")
            results[model_name] = {'adj_r2': 0, 'best_model': None, 'val_adj_r2': 0, 'best_params': {}}
    stock_results[stock_name] = results

# Step 6.5: Analyzing overfitting...
print("\nStep 6.5: Analyzing overfitting...")
for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    for model_name, result in results.items():
        best_model = result['best_model']
        if best_model is None:
            continue
        data_type = next((m[2] for m in models if m[0] == model_name), None)
        try:
            model_temp = model_fn(result['best_params'])
            if data_type == 'ml':
                train_X_2d = train_X.reshape(train_X.shape[0], -1)
                val_X_2d = val_X.reshape(val_X.shape[0], -1)
                model_temp.fit(train_X_2d, train_y)
                train_pred = model_temp.predict(train_X_2d)
                val_pred = model_temp.predict(val_X_2d)
            else:
                model_temp.fit(train_X, train_y, epochs=100, batch_size=64, validation_data=(val_X, val_y), verbose=0,
                               callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])
                train_pred = model_temp.predict(train_X, verbose=0).flatten()
                val_pred = model_temp.predict(val_X, verbose=0).flatten()

            train_y_inv = price_scaler.inverse_transform(train_y.reshape(-1, 1)).flatten()
            val_y_inv = price_scaler.inverse_transform(val_y.reshape(-1, 1)).flatten()
            train_pred_inv = price_scaler.inverse_transform(train_pred.reshape(-1, 1)).flatten()
            val_pred_inv = price_scaler.inverse_transform(val_pred.reshape(-1, 1)).flatten()

            train_adj_r2 = calculate_adj_r2(train_y_inv, train_pred_inv, feature_dim)
            val_adj_r2 = calculate_adj_r2(val_y_inv, val_pred_inv, feature_dim)
            overfitting_gap = train_adj_r2 - val_adj_r2

            stock_results[stock_name][model_name]['train_adj_r2'] = train_adj_r2
            stock_results[stock_name][model_name]['overfitting_gap'] = overfitting_gap
            print(f"Model: {model_name} for {stock_name} - Val Adj R²: {val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap:.6f}")
            print(f"  Training Adj R²: {train_adj_r2:.6f}")
            if overfitting_gap > 0.2:
                print("  Warning: Potential overfitting detected")
            elif overfitting_gap > 0.1:
                print("  Caution: Moderate overfitting risk")
            else:
                print("  Status: Low overfitting risk")
        except Exception as e:
            print(f"Error analyzing overfitting for {model_name}: {e}")

[I 2025-12-26 12:16:19,825] A new study created in memory with name: no-name-a8802292-c664-42e6-a5be-49462154bd79



Step 6: Training and evaluating models...

Training cnn_lstm for UBER...


[I 2025-12-26 12:17:11,152] Trial 0 finished with value: -0.336736755899665 and parameters: {'filters': 43, 'lstm_units': 126, 'dropout_rate': 0.36095683201628315, 'learning_rate': 0.004476090836378951}. Best is trial 0 with value: -0.336736755899665.
[I 2025-12-26 12:19:00,832] Trial 1 finished with value: -0.5682505039165139 and parameters: {'filters': 62, 'lstm_units': 85, 'dropout_rate': 0.47872547402714427, 'learning_rate': 0.00348632707414647}. Best is trial 1 with value: -0.5682505039165139.
[I 2025-12-26 12:20:51,266] Trial 2 finished with value: -0.6126643976644872 and parameters: {'filters': 50, 'lstm_units': 99, 'dropout_rate': 0.5583719393521851, 'learning_rate': 0.00047381603895325827}. Best is trial 2 with value: -0.6126643976644872.
[I 2025-12-26 12:22:55,235] Trial 3 finished with value: -0.6102121434126693 and parameters: {'filters': 60, 'lstm_units': 99, 'dropout_rate': 0.37082633931060505, 'learning_rate': 0.0012683015088902}. Best is trial 2 with value: -0.612664397

cnn_lstm - UBER - Val Adj R²: 0.705526, Test Adj R²: -1.674719

Training conv1d_lstm for UBER...


[I 2025-12-26 12:34:52,424] Trial 0 finished with value: -0.512141011308374 and parameters: {'filters': 64, 'lstm_units': 85, 'dropout_rate': 0.3854415256464313, 'learning_rate': 0.0010854886886268033}. Best is trial 0 with value: -0.512141011308374.
[I 2025-12-26 12:35:31,166] Trial 1 finished with value: -0.2956986130665975 and parameters: {'filters': 42, 'lstm_units': 55, 'dropout_rate': 0.5851486335242704, 'learning_rate': 0.003489555320445505}. Best is trial 0 with value: -0.512141011308374.
[I 2025-12-26 12:36:17,127] Trial 2 finished with value: -0.7634893339588362 and parameters: {'filters': 41, 'lstm_units': 79, 'dropout_rate': 0.36346023900998264, 'learning_rate': 0.0035190402626219647}. Best is trial 2 with value: -0.7634893339588362.
[I 2025-12-26 12:36:56,115] Trial 3 finished with value: -0.7516381845376995 and parameters: {'filters': 33, 'lstm_units': 91, 'dropout_rate': 0.4248561452332478, 'learning_rate': 0.004578364870823044}. Best is trial 2 with value: -0.7634893339

...

xgboost - UBER - Val Adj R²: 0.962551, Test Adj R²: 0.990387

Training transformer for UBER...


[I 2025-12-26 13:31:20,498] Trial 0 finished with value: -0.7987218499847704 and parameters: {'num_heads': 8, 'ff_dim': 68, 'dropout_rate': 0.4958960777236221, 'learning_rate': 0.0005426594570823743}. Best is trial 0 with value: -0.7987218499847704.
[I 2025-12-26 13:31:40,519] Trial 1 finished with value: -0.8030948666355413 and parameters: {'num_heads': 4, 'ff_dim': 87, 'dropout_rate': 0.48597413078977686, 'learning_rate': 0.0017733607827648107}. Best is trial 1 with value: -0.8030948666355413.
[I 2025-12-26 13:31:59,243] Trial 2 finished with value: -0.7704953303074196 and parameters: {'num_heads': 5, 'ff_dim': 114, 'dropout_rate': 0.4164382372597161, 'learning_rate': 0.004874884018516449}. Best is trial 1 with value: -0.8030948666355413.
[I 2025-12-26 13:32:15,369] Trial 3 finished with value: -0.07288483282794744 and parameters: {'num_heads': 4, 'ff_dim': 98, 'dropout_rate': 0.5270498930087353, 'learning_rate': 0.004949965329545407}. Best is trial 1 with value: -0.8030948666355413.

transformer - UBER - Val Adj R²: 0.818097, Test Adj R²: -47.923235

Training tcn for UBER...


[I 2025-12-26 13:35:13,771] Trial 0 finished with value: -0.8471292022352064 and parameters: {'filters': 41, 'kernel_size': 5, 'dropout_rate': 0.52772936836357, 'learning_rate': 0.00100176461609772}. Best is trial 0 with value: -0.8471292022352064.
[I 2025-12-26 13:35:51,604] Trial 1 finished with value: -0.8363550318254598 and parameters: {'filters': 47, 'kernel_size': 3, 'dropout_rate': 0.583586976700345, 'learning_rate': 0.00017874960337229358}. Best is trial 0 with value: -0.8471292022352064.
[I 2025-12-26 13:36:24,651] Trial 2 finished with value: -0.827977152890381 and parameters: {'filters': 62, 'kernel_size': 3, 'dropout_rate': 0.43141027641342905, 'learning_rate': 0.0029137413144417184}. Best is trial 0 with value: -0.8471292022352064.
[I 2025-12-26 13:36:47,888] Trial 3 finished with value: -0.8213751865089222 and parameters: {'filters': 46, 'kernel_size': 4, 'dropout_rate': 0.42273762387854646, 'learning_rate': 0.0030161945240373654}. Best is trial 0 with value: -0.847129202

tcn - UBER - Val Adj R²: 0.866840, Test Adj R²: 0.942351

Step 6.5: Analyzing overfitting...
Error analyzing overfitting for cnn_lstm: 'kernel_size'
Error analyzing overfitting for conv1d_lstm: 'kernel_size'
Error analyzing overfitting for bilstm: 'filters'
Error analyzing overfitting for stacked_lstm: 'filters'
Error analyzing overfitting for gru: 'filters'
Error analyzing overfitting for lightgbm: 'filters'
Error analyzing overfitting for xgboost: 'filters'
Error analyzing overfitting for transformer: 'filters'
Model: tcn for UBER - Val Adj R²: 0.856740, Overfitting Gap: 0.140248
  Training Adj R²: 0.996988
  Caution: Moderate overfitting risk


In [ ]:
# Step 7: Generate Predictions with Best Non-Overfit Model
print("\nStep 7: Generating predictions...")
best_val_adj_r2 = -float('inf')
best_model_name = None
best_model = None

for stock_name in stock_results.keys():
    results = stock_results[stock_name]
    for model_name, result in results.items():
        if result.get('val_adj_r2', -float('inf')) > best_val_adj_r2 and result.get('overfitting_gap', float('inf')) <= 0.2:
            best_val_adj_r2 = result['val_adj_r2']
            best_model_name = model_name
            best_model = result['best_model']

if best_model_name is None:
    print("No model with acceptable overfitting gap found. Using best overall Val Adj R².")
    best_model_name = max(results.items(), key=lambda x: x[1]['val_adj_r2'])[0]
    best_model = results[best_model_name]['best_model']

overfitting_gap = results[best_model_name].get('overfitting_gap', 'N/A')
print(f"Best model: {best_model_name} with Val Adj R²: {best_val_adj_r2:.6f}, Overfitting Gap: {overfitting_gap}")

def generate_predictions(model, model_name, scaled_data, price_scaler, stock_name, timestamps, steps=150):
    preds_scaled = []
    current_sequence = scaled_data[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, feature_dim)
    for _ in range(steps):
        try:
            if model_name in ['xgboost', 'lightgbm']:
                pred_scaled = model.predict(current_sequence.reshape(1, -1))[0]
            else:
                pred_scaled = model.predict(current_sequence, verbose=0)[0, 0]
            preds_scaled.append(float(pred_scaled))
            current_sequence = np.roll(current_sequence, -1, axis=1)
            current_sequence[0, -1, 0] = pred_scaled
            if feature_dim > 1:
                current_sequence[0, -1, 1:] = current_sequence[0, -2, 1:]
        except Exception as e:
            print(f"Prediction failed for {model_name}: {e}")
            break
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    predictions = price_scaler.inverse_transform(preds_scaled).flatten()
    last_timestamp = pd.to_datetime(timestamps.iloc[-1])
    future_timestamps = pd.date_range(start=last_timestamp + pd.Timedelta(minutes=1), periods=steps, freq='T')
    return predictions, future_timestamps

predictions, future_timestamps = generate_predictions(best_model, best_model_name, last_300_scaled, price_scaler, stock_name, stock_data[TARGET_COLUMN]['Timestamp'])
print(f"Predictions (first 5): {predictions[:5]}")
print(f"Future timestamps (first 5): {future_timestamps[:5]}")


Step 7: Generating predictions...
Best model: tcn with Val Adj R²: 0.866840, Overfitting Gap: 0.14024792223355065
Predictions (first 5): [81.09288963 81.09071544 81.0886959  81.08682587 81.08514009]
Future timestamps (first 5): DatetimeIndex(['2025-12-24 17:46:00+00:00', '2025-12-24 17:47:00+00:00',
               '2025-12-24 17:48:00+00:00', '2025-12-24 17:49:00+00:00',
               '2025-12-24 17:50:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='min')


In [ ]:
# Step 8: Filter Models and Generate Predictions
print("\nStep 8: Filtering models and generating predictions for those with Adj R² > 0.6...")

filtered_models = {stock_name: {} for stock_name in stock_results.keys()}
adj_r2_threshold = 0.6

for stock_name, results in stock_results.items():
    print(f"\nProcessing models for {stock_name}...")
    for model_name, result in results.items():
        test_adj_r2 = result.get('adj_r2', -float('inf'))
        if test_adj_r2 > adj_r2_threshold:
            print(f"  {model_name} has Test Adj R² of {test_adj_r2:.6f} (>{adj_r2_threshold}). Generating predictions...")
            best_model = result['best_model']
            if best_model is not None:
                try:
                    predictions, future_timestamps = generate_predictions(best_model, model_name, last_300_scaled, price_scaler, stock_name, stock_data[TARGET_COLUMN]['Timestamp'])
                    prediction_df = pd.DataFrame({
                        'Timestamp': future_timestamps,
                        f'{stock_name}_Predicted_Price': predictions
                    })
                    filtered_models[stock_name][model_name] = prediction_df
                    print(f"  Generated {len(predictions)} predictions for {model_name}.")
                except Exception as e:
                    print(f"  Error generating predictions for {model_name}: {e}")
            else:
                print(f"  Skipping {model_name} due to missing model.")
        else:
            print(f"  {model_name} has Test Adj R² of {test_adj_r2:.6f} (<={adj_r2_threshold}). Skipping.")

# Step 9: Download Predictions as CSV
print("\nStep 9: Downloading predictions as CSV...")

for stock_name, models_predictions in filtered_models.items():
    if not models_predictions:
        print(f"No models with Adj R² > {adj_r2_threshold} for {stock_name}. Skipping download.")
        continue

    for model_name, prediction_df in models_predictions.items():
        file_name = f'{stock_name}_{model_name}_predictions.csv'
        try:
            prediction_df.to_csv(file_name, index=False)
            print(f"Saved predictions for {stock_name} using {model_name} to {file_name}")
        except Exception as e:
            print(f"Error saving {file_name}: {e}")

print("\nPrediction generation and download complete.")


Step 8: Filtering models and generating predictions for those with Adj R² > 0.6...

Processing models for UBER...
  cnn_lstm has Test Adj R² of -1.674719 (<=0.6). Skipping.
  conv1d_lstm has Test Adj R² of -0.385903 (<=0.6). Skipping.
  bilstm has Test Adj R² of 0.928877 (>0.6). Generating predictions...
  Generated 150 predictions for bilstm.
  stacked_lstm has Test Adj R² of 0.862136 (>0.6). Generating predictions...
  Generated 150 predictions for stacked_lstm.
  gru has Test Adj R² of 0.678355 (>0.6). Generating predictions...
  Generated 150 predictions for gru.
  lightgbm has Test Adj R² of 0.989436 (>0.6). Generating predictions...
  Generated 150 predictions for lightgbm.
  xgboost has Test Adj R² of 0.990387 (>0.6). Generating predictions...
  Generated 150 predictions for xgboost.
  transformer has Test Adj R² of -47.923235 (<=0.6). Skipping.
  tcn has Test Adj R² of 0.942351 (>0.6). Generating predictions...
  Generated 150 predictions for tcn.

Step 9: Downloading predicti

In [ ]:
# Step 10: Download prediction files to local computer
print("\nStep 10: Downloading prediction files to your computer...")

try:
    for stock_name, models_predictions in filtered_models.items():
        for model_name, prediction_df in models_predictions.items():
            file_name = f'{stock_name}_{model_name}_predictions.csv'
            files.download(file_name)
            print(f"Downloaded {file_name}")
except Exception as e:
    print(f"Error during file download: {e}")

print("\nDownload process complete.")


Step 10: Downloading prediction files to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded UBER_bilstm_predictions.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded UBER_xgboost_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded UBER_tcn_predictions.csv

Download process complete.
